In [5]:
import torch
import torch.nn as nn

class BinaryClassifier(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2, hidden_layers=[256, 128, 64]):
        super().__init__()

        layerlist = []
        n_in = input_size

        for h in hidden_layers:
            layerlist += [
                nn.Linear(n_in, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),
            ]
            n_in = h

        # Final output layer: 1 logit for binary classification
        layerlist.append(nn.Linear(n_in, 1))

        self.network = nn.Sequential(*layerlist)

        # Better initialization (same as before)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        # shape: (batch_size, 1) → squeeze to (batch_size,)
        logits = self.network(x).squeeze(-1)
        return logits


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 0) Load ALREADY-SCALED binary dataset
df_binary = pd.read_csv("artifacts/final_train_scaled_binary_clustered.csv")

TARGET_COL = "MP_binary_high"

# Exclude non-features
exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}

# Use only numeric feature columns
num_cols = df_binary.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

# Build X/y
X = df_binary[feature_cols].to_numpy(np.float32)
y = df_binary[TARGET_COL].to_numpy(np.float32)

# Stratification labels (cluster)
y_strat = df_binary["Structure_Cluster"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("n features:", len(feature_cols))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

# 1) Fix folds once
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr_idx, val_idx) for tr_idx, val_idx in skf.split(X, y_strat)]
print("Built folds:", len(folds))


X shape: (17625, 202)
y shape: (17625,)
n features: 202
Label counts: {np.float32(0.0): np.int64(16853), np.float32(1.0): np.int64(772)}
Built folds: 10


In [7]:
from sklearn.metrics import accuracy_score
from pathlib import Path

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    

from scipy.special import expit  # stable sigmoid

def save_checkpoint(
    model,
    optimizer,
    epoch,
    train_loss,
    val_loss,
    y_train,
    y_val,
    val_loader,
    fold_idx,
    checkpoint_dir,
    checkpoint_tracking,
    is_final=False,
):
    # Put model in eval mode and collect logits
    model.eval()
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            logits = model(xb).cpu().numpy()
            all_logits.append(logits)

    logits_val = np.concatenate(all_logits)   # shape (N,)
    probs_val  = expit(logits_val)            # stable sigmoid
    preds_val  = (probs_val >= 0.5).astype(int)

    # y_val expected as numpy 0/1
    y_val_np = np.asarray(y_val).astype(int)

    # Classification metric: accuracy
    acc = accuracy_score(y_val_np, preds_val)

    # Filename
    if is_final:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
    else:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"

    checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename

    # Save checkpoint (weights + optimizer + losses + metric)
    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc,
        "fold_idx": fold_idx,
        "is_final": is_final,
    }
    torch.save(checkpoint_data, checkpoint_path_full)

    # Track info for CSV
    checkpoint_info = {
        "Fold": fold_idx,
        "Epoch": epoch,
        "Checkpoint_Filename": checkpoint_filename,
        "Checkpoint_Path": str(checkpoint_path_full),
        "Train_Loss": round(train_loss, 6),
        "Val_Loss": round(val_loss, 6),
        "Accuracy": round(acc, 6),
        "Is_Final": is_final,
    }
    checkpoint_tracking.append(checkpoint_info)

    checkpoint_type = "Final" if is_final else "Regular"
    print(
        f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - "
        f"Val Loss: {val_loss:.4f} | Acc: {acc:.4f}"
    )
    return True


In [ ]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SVMSMOTE


def evaluate_fold(
    trial,
    fold_idx,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    hidden_layers,
    learning_rate,
    batch_size,
    dropout_rate,
    weight_decay,
    max_epochs=10**9,
    patience=30,
    min_delta=0,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_classifier",
    save_every_n_epochs=15,
    use_svmsmote=True,              # <-- NEW (optional)
    svmsmote_kwargs=None,           # <-- NEW (optional)
):

    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu (binary classifier, BCELoss)")

    # ---- checkpoints ----
    checkpoint_tracking = []
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # ==========================================================
    # Apply SVM-SMOTE on TRAIN ONLY (never on validation)
    # ==========================================================
    if use_svmsmote:
        svmsmote_kwargs = svmsmote_kwargs or {}
        smote = SVMSMOTE(random_state=42, **svmsmote_kwargs)

        # 1) SMOTE expects class labels as integers (0/1)
        y_train_int = np.asarray(y_train).astype(np.int64)

        # 2) Resample
        X_train_scaled, y_train_int = smote.fit_resample(X_train_scaled, y_train_int)

        # 3) Convert back to float for BCELoss
        y_train = y_train_int.astype(np.float32)

        print(
            f"[Fold {fold_idx}] After SVM-SMOTE: "
            f"X_train={X_train_scaled.shape}, "
            f"counts={np.bincount(y_train_int)}"
        )
    else:
        # Ensure float for BCELoss
        y_train = np.asarray(y_train).astype(np.float32)


    # ---- tensors (y must be float 0/1) ----
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # ---- model + BCELoss ----
    model = BinaryClassifier(
        input_size=X_train_scaled.shape[1],
        hidden_layers=hidden_layers,
        dropout_rate=dropout_rate,
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

    # ---- training loop ----
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            logits = model(xb)                         # (batch,)
            probs  = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)   # <-- recommended with BCELoss
            loss   = criterion(probs, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb)
                probs  = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)  # <-- same clamp
                loss   = criterion(probs, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # checkpoints
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader,   # y_train changed by SMOTE, y_val unchanged
                fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                    is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # ---- load best ----
    model.load_state_dict(best_state)
    model.eval()

    # save checkpoint tracking csv
    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_classifier.csv", index=False)

    # ---- final accuracy on val set (best model) ----
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_logits.append(model(xb).cpu().numpy())
    logits_val = np.concatenate(all_logits)
    probs_val  = 1 / (1 + np.exp(-logits_val))
    preds_val  = (probs_val >= 0.5).astype(int)

    y_val_np = np.asarray(y_val).astype(int)
    acc = accuracy_score(y_val_np, preds_val)

    return best_val_loss, acc, model, train_losses, val_losses, stop_epoch


In [12]:
import time
import numpy as np
from pathlib import Path

trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # Hidden layers (same logic)
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    val_losses = []
    accs = []

    # Run this hyperparameter combo across all folds
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]

        # Put classifier checkpoints in a classifier-specific folder
        trial_checkpoint_root = Path("checkpoints_binary_classifier") / f"trial_{trial.number:03d}"

        best_val_loss, acc, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=False,
            checkpoint_dir=trial_checkpoint_root,
        )

        val_losses.append(best_val_loss)
        accs.append(acc)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_val_loss = float(np.mean(val_losses))
    avg_acc      = float(np.mean(accs))

    print(f"Trial {trial.number}: Avg Val Loss = {avg_val_loss:.4f} | Avg Acc = {avg_acc:.4f}")

    # Optuna minimizes this
    return avg_val_loss

In [13]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print(best_params)

[I 2025-12-22 00:04:48,202] A new study created in memory with name: no-name-9bd560ed-03f9-4e94-bbb9-6ccaf68d8d0e


Fold 0: Training on cpu (binary classifier, BCELoss)
[Fold 0] After SVM-SMOTE: X_train=(30334, 202), counts=[15167 15167]
[Fold 0] Epoch    1 | Train Loss: 0.5149 | Val Loss: 0.3669 | ES 0/30
[Fold 0] Early stopping at epoch 50 (best Val Loss: 0.1625)
Fold 1: Training on cpu (binary classifier, BCELoss)
[Fold 1] After SVM-SMOTE: X_train=(30320, 202), counts=[15160 15160]
[Fold 1] Epoch    1 | Train Loss: 0.4973 | Val Loss: 0.2756 | ES 0/30
[Fold 1] Epoch   50 | Train Loss: 0.1192 | Val Loss: 0.1764 | ES 28/30
[Fold 1] Early stopping at epoch 52 (best Val Loss: 0.1629)
Fold 2: Training on cpu (binary classifier, BCELoss)
[Fold 2] After SVM-SMOTE: X_train=(30336, 202), counts=[15168 15168]
[Fold 2] Epoch    1 | Train Loss: 0.4895 | Val Loss: 0.2828 | ES 0/30
[Fold 2] Epoch   50 | Train Loss: 0.1111 | Val Loss: 0.2083 | ES 24/30
[Fold 2] Early stopping at epoch 56 (best Val Loss: 0.1705)
Fold 3: Training on cpu (binary classifier, BCELoss)
[Fold 3] After SVM-SMOTE: X_train=(30304, 202), c

[I 2025-12-22 00:35:48,880] Trial 0 finished with value: 0.16755042306200008 and parameters: {'dropout_rate': 0.416182338059732, 'learning_rate': 0.00010490405719191396, 'weight_decay': 1.8147695894049015e-06, 'batch_size': 32, 'h1': 256}. Best is trial 0 with value: 0.16755042306200008.


[Fold 9] Early stopping at epoch 47 (best Val Loss: 0.1528)
Trial 0 finished in 31.01 minutes
Trial 0: Avg Val Loss = 0.1676 | Avg Acc = 0.9430
Fold 0: Training on cpu (binary classifier, BCELoss)
[Fold 0] After SVM-SMOTE: X_train=(30334, 202), counts=[15167 15167]
[Fold 0] Epoch    1 | Train Loss: 0.3075 | Val Loss: 0.2034 | ES 0/30
[Fold 0] Early stopping at epoch 47 (best Val Loss: 0.1588)
Fold 1: Training on cpu (binary classifier, BCELoss)
[Fold 1] After SVM-SMOTE: X_train=(30320, 202), counts=[15160 15160]
[Fold 1] Epoch    1 | Train Loss: 0.2915 | Val Loss: 0.2382 | ES 0/30
[Fold 1] Early stopping at epoch 42 (best Val Loss: 0.1561)
Fold 2: Training on cpu (binary classifier, BCELoss)
[Fold 2] After SVM-SMOTE: X_train=(30336, 202), counts=[15168 15168]
[Fold 2] Epoch    1 | Train Loss: 0.3031 | Val Loss: 0.2179 | ES 0/30
[Fold 2] Early stopping at epoch 37 (best Val Loss: 0.1610)
Fold 3: Training on cpu (binary classifier, BCELoss)
[Fold 3] After SVM-SMOTE: X_train=(30304, 202),

[I 2025-12-22 00:48:23,093] Trial 1 finished with value: 0.1575937094582644 and parameters: {'dropout_rate': 0.32941113696776936, 'learning_rate': 0.0008526280268138465, 'weight_decay': 1.5745199751107903e-06, 'batch_size': 64, 'h1': 192}. Best is trial 1 with value: 0.1575937094582644.


...

[Fold 9] Early stopping at epoch 66 (best Val Loss: 0.1377)
Trial 17 finished in 14.08 minutes
Trial 17: Avg Val Loss = 0.1580 | Avg Acc = 0.9468
Fold 0: Training on cpu (binary classifier, BCELoss)
[Fold 0] After SVM-SMOTE: X_train=(30334, 202), counts=[15167 15167]
[Fold 0] Epoch    1 | Train Loss: 0.4230 | Val Loss: 0.2392 | ES 0/30
[Fold 0] Early stopping at epoch 49 (best Val Loss: 0.1431)
Fold 1: Training on cpu (binary classifier, BCELoss)
[Fold 1] After SVM-SMOTE: X_train=(30320, 202), counts=[15160 15160]
[Fold 1] Epoch    1 | Train Loss: 0.4212 | Val Loss: 0.2551 | ES 0/30
[Fold 1] Early stopping at epoch 47 (best Val Loss: 0.1558)
Fold 2: Training on cpu (binary classifier, BCELoss)
[Fold 2] After SVM-SMOTE: X_train=(30336, 202), counts=[15168 15168]
[Fold 2] Epoch    1 | Train Loss: 0.4197 | Val Loss: 0.2417 | ES 0/30
[Fold 2] Epoch   50 | Train Loss: 0.0619 | Val Loss: 0.1917 | ES 25/30
[Fold 2] Early stopping at epoch 55 (best Val Loss: 0.1662)
Fold 3: Training on cpu (bi

[I 2025-12-22 05:05:07,534] Trial 18 finished with value: 0.1543004543387464 and parameters: {'dropout_rate': 0.4444704208987915, 'learning_rate': 0.0006730442199307018, 'weight_decay': 1.0678628339728529e-06, 'batch_size': 64, 'h1': 128}. Best is trial 11 with value: 0.151469586690655.


[Fold 9] Early stopping at epoch 48 (best Val Loss: 0.1409)
Trial 18 finished in 10.24 minutes
Trial 18: Avg Val Loss = 0.1543 | Avg Acc = 0.9477
Fold 0: Training on cpu (binary classifier, BCELoss)
[Fold 0] After SVM-SMOTE: X_train=(30334, 202), counts=[15167 15167]
[Fold 0] Epoch    1 | Train Loss: 0.7379 | Val Loss: 0.6582 | ES 0/30
[Fold 0] Epoch   50 | Train Loss: 0.2558 | Val Loss: 0.2541 | ES 8/30
[Fold 0] Epoch  100 | Train Loss: 0.2388 | Val Loss: 0.2349 | ES 11/30
[Fold 0] Early stopping at epoch 119 (best Val Loss: 0.2076)
Fold 1: Training on cpu (binary classifier, BCELoss)
[Fold 1] After SVM-SMOTE: X_train=(30320, 202), counts=[15160 15160]
[Fold 1] Epoch    1 | Train Loss: 0.7673 | Val Loss: 0.5826 | ES 0/30
[Fold 1] Epoch   50 | Train Loss: 0.2602 | Val Loss: 0.2525 | ES 12/30
[Fold 1] Epoch  100 | Train Loss: 0.2376 | Val Loss: 0.2574 | ES 1/30
[Fold 1] Early stopping at epoch 139 (best Val Loss: 0.2056)
Fold 2: Training on cpu (binary classifier, BCELoss)
[Fold 2] Afte

[I 2025-12-22 05:37:39,064] Trial 19 finished with value: 0.19045102364782776 and parameters: {'dropout_rate': 0.3791143003788505, 'learning_rate': 1.2889109673934805e-05, 'weight_decay': 2.8640178488818598e-05, 'batch_size': 64, 'h1': 160}. Best is trial 11 with value: 0.151469586690655.


[Fold 9] Early stopping at epoch 92 (best Val Loss: 0.1819)
Trial 19 finished in 32.53 minutes
Trial 19: Avg Val Loss = 0.1905 | Avg Acc = 0.9181
{'dropout_rate': 0.48029122880638786, 'learning_rate': 0.0009887000767176209, 'weight_decay': 1.096865011667428e-06, 'batch_size': 64, 'h1': 128}


In [14]:
print("Best Trial Number:", study.best_trial.number)
print(best_params)

Best Trial Number: 11
{'dropout_rate': 0.48029122880638786, 'learning_rate': 0.0009887000767176209, 'weight_decay': 1.096865011667428e-06, 'batch_size': 64, 'h1': 128}


In [15]:
from pathlib import Path
import torch
import pandas as pd

# BASE and artifacts_dir should already be defined (same script as before)
BASE = Path.cwd()  # transfer_learning
artifacts_dir = BASE / "artifacts"

# ---------- Directories for final best models + checkpoints ----------
best_models_dir = artifacts_dir / "binary_SVM_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_binary_SVM_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

print("Best hyperparameters from Optuna:", best_params)

# Helper to derive hidden layers from best_params (same logic as in objective)
def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay, "| batch_size:", batch_size)

final_fold_metrics = []

# ---------- Final training loop for all folds (using `folds`, X, y) ----------
# Assumes you already defined:
#   X, y, folds = [(tr_idx, val_idx), ...] earlier (same as in `objective`)
for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final training for fold {fold_idx} ====================")

    X_train_scaled = X[tr_idx]
    y_train        = y[tr_idx]
    X_val_scaled   = X[val_idx]
    y_val          = y[val_idx]

    # Train this fold with the best hyperparameters
    best_val_loss, acc, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,
        fold_idx=fold_idx,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,
        checkpoint_dir=final_ckpt_dir,   # evaluate_fold will make fold_{k}/ inside
        save_every_n_epochs=15,
    )

    # ---------- Save the final (best) model for this fold ----------
    model_path = best_models_dir / f"binary_best_fold_{fold_idx}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "fold_idx": fold_idx,
            "best_val_bce": float(best_val_loss),
            "val_accuracy": float(acc),
        },
        model_path,
    )
    print(f"[Fold {fold_idx}] Saved best classifier model to: {model_path}")
    print(f"[Fold {fold_idx}] Val BCE: {best_val_loss:.4f} | Val Acc: {acc:.4f} | Stop epoch: {stop_epoch}")

    # store metrics
    final_fold_metrics.append(
        {
            "Fold": fold_idx,
            "Best_Val_BCE": float(best_val_loss),
            "Val_Accuracy": float(acc),
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

# ---------- Save a summary CSV of all folds ----------
metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "binary_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved summary of best classifier models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'dropout_rate': 0.48029122880638786, 'learning_rate': 0.0009887000767176209, 'weight_decay': 1.096865011667428e-06, 'batch_size': 64, 'h1': 128}
Using hidden_layers: [128, 64, 32]
dropout: 0.48029122880638786 | lr: 0.0009887000767176209 | wd: 1.096865011667428e-06 | batch_size: 64

==================== Final training for fold 0 ====================
Fold 0: Training on cpu (binary classifier, BCELoss)
Checkpoints will be saved to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/checkpoints_binary_SVM_best/fold_0
[Fold 0] After SVM-SMOTE: X_train=(30334, 202), counts=[15167 15167]
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.2227 | Acc: 0.8990
[Fold 0] Epoch    1 | Train Loss: 0.4067 | Val Loss: 0.2227 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.1842 | Acc: 0.9359
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.1561 | Acc: 0.9535
[Fold 0] Early stopping at epoch 41 (best Val Loss: 0.1491)
[Fo

In [18]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit

# ---------- Directories for final best models + checkpoints ----------
best_models_dir = artifacts_dir / "binary_SVM_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_binary_SVM_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)


confusion_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Confusion matrix for fold {fold_idx} =====")

    # Load validation data
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # Load best model
    model_path = best_models_dir / f"binary_best_fold_{fold_idx}.pt"
    checkpoint = torch.load(model_path, map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=checkpoint["hidden_layers"],
        dropout_rate=checkpoint["dropout_rate"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Predict
    with torch.no_grad():
        logits = model(X_val_tensor).cpu().numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    confusion_matrices.append(cm)

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")




===== Confusion matrix for fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1625   61]
 [  27   50]]
TN=1625, FP=61, FN=27, TP=50

===== Confusion matrix for fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1643   50]
 [  31   39]]
TN=1643, FP=50, FN=31, TP=39

===== Confusion matrix for fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1617   68]
 [  26   52]]
TN=1617, FP=68, FN=26, TP=52

===== Confusion matrix for fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1670   31]
 [  31   31]]
TN=1670, FP=31, FN=31, TP=31

===== Confusion matrix for fold 4 =====
Confusion matrix (rows=true, cols=pred):
[[1604   70]
 [  35   54]]
TN=1604, FP=70, FN=35, TP=54

===== Confusion matrix for fold 5 =====
Confusion matrix (rows=true, cols=pred):
[[1628   59]
 [  33   42]]
TN=1628, FP=59, FN=33, TP=42

===== Confusion matrix for fold 6 =====
Confusion matrix (rows=true, cols=pred):
[[1585   76]
 [  31   70]]
TN=1585, FP=76, FN=31, TP=70

===== Confusion matrix for fold 7

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_62264/4167719322.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_loca

In [21]:
mean_cm = np.mean(confusion_matrices, axis=0)

mean_cm_df = pd.DataFrame(
    mean_cm,
    index=["True Low&Int", "True High"],
    columns=["Pred Low&Int", "Pred High"],
)

print("\n===== Mean confusion matrix across folds =====")
print(mean_cm_df)

# Save to CSV
mean_cm_path = best_models_dir / "binary_mean_confusion_matrix.csv"
mean_cm_df.to_csv(mean_cm_path)
print(f"\n✅ Saved mean confusion matrix to: {mean_cm_path}")



===== Mean confusion matrix across folds =====
              Pred Low&Int  Pred High
True Low&Int        1629.7       55.6
True High             31.2       46.0

✅ Saved mean confusion matrix to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/artifacts/binary_SVM_best_models/binary_mean_confusion_matrix.csv


In [19]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit  # stable sigmoid

def metrics_from_confusion(cm):
    """
    cm = [[TN, FP],
          [FN, TP]]
    """
    TN, FP, FN, TP = cm.ravel()

    accuracy  = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Specificity": specificity,
    }

all_fold_metrics = []
all_conf_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\nEvaluating fold {fold_idx}")

    # Load best model for this fold
    ckpt = torch.load(best_models_dir / f"binary_best_fold_{fold_idx}.pt", map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=ckpt["hidden_layers"],
        dropout_rate=ckpt["dropout_rate"],
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Validation data
    X_val = torch.tensor(X[val_idx], dtype=torch.float32)
    y_val = y[val_idx].astype(int)

    # Predict
    with torch.no_grad():
        logits = model(X_val).numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # Confusion matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_val, preds)
    all_conf_matrices.append(cm)

    # Metrics
    metrics = metrics_from_confusion(cm)
    metrics["Fold"] = fold_idx
    all_fold_metrics.append(metrics)

    print("Confusion matrix:\n", cm)
    print(metrics)



Evaluating fold 0
Confusion matrix:
 [[1625   61]
 [  27   50]]
{'Accuracy': np.float64(0.9500850822461713), 'Precision': np.float64(0.45045045045045046), 'Recall': np.float64(0.6493506493506493), 'F1': np.float64(0.5319148936170213), 'Specificity': np.float64(0.9638196915776986), 'Fold': 0}

Evaluating fold 1
Confusion matrix:
 [[1643   50]
 [  31   39]]
{'Accuracy': np.float64(0.9540555870674986), 'Precision': np.float64(0.43820224719101125), 'Recall': np.float64(0.5571428571428572), 'F1': np.float64(0.4905660377358491), 'Specificity': np.float64(0.9704666272888364), 'Fold': 1}

Evaluating fold 2
Confusion matrix:
 [[1617   68]
 [  26   52]]
{'Accuracy': np.float64(0.9466817923993194), 'Precision': np.float64(0.43333333333333335), 'Recall': np.float64(0.6666666666666666), 'F1': np.float64(0.5252525252525252), 'Specificity': np.float64(0.9596439169139466), 'Fold': 2}

Evaluating fold 3
Confusion matrix:
 [[1670   31]
 [  31   31]]
{'Accuracy': np.float64(0.9648326715825298), 'Precisi

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_62264/1914565896.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(best_models_dir / f"binary

In [20]:
metrics_df = pd.DataFrame(all_fold_metrics)

mean_metrics = metrics_df.mean(numeric_only=True)
std_metrics  = metrics_df.std(numeric_only=True)

summary_df = pd.DataFrame({
    "Mean": mean_metrics,
    "Std": std_metrics
})

print("\n=== Cross-validated performance ===")
print(summary_df)

summary_df.to_csv(
    best_models_dir / "binary_classifier_confusion_metrics_summary.csv"
)



=== Cross-validated performance ===
                 Mean       Std
Accuracy     0.950752  0.008200
Precision    0.455377  0.027454
Recall       0.588711  0.067921
F1           0.510994  0.025737
Specificity  0.966961  0.008975
Fold         4.500000  3.027650
